# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Samia2310/flyrank-assignment1-week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content page for one client on one report date. For this assignment I analyze only rows where month='2026-03'.

For this assignment, I will use data from March 2026 (month = 2026-03) because it is a mid-panel month that avoids using the final month as training data.

The goal is to score each webpage according to its content opportunity so that pages can be prioritized for actions such as refresh, expansion, monitoring, or protection.

In [1]:
!pip uninstall -y duckdb
!pip install -U duckdb

Found existing installation: duckdb 1.5.4
Uninstalling duckdb-1.5.4:
  Successfully uninstalled duckdb-1.5.4
  Using cached duckdb-1.5.4-cp312-cp312-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl.metadata (4.2 kB)
Using cached duckdb-1.5.4-cp312-cp312-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl (21.5 MB)


In [2]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [3]:
query = """
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
LIMIT 10
"""

df = con.execute(query).df()
df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Feature              | Available when?                                                                    |
| -------------------- | ---------------------------------------------------------------------------------- |
| **gsc_clicks**       | Known at the decision moment because it comes from historical Search Console data. |
| **gsc_impressions**  | Known because impressions have already been recorded before making a prediction.   |
| **gsc_avg_position** | Known because historical ranking data already exists.                              |
| **ga4_sessions**     | Known because past session data is available before prediction.                    |
| **ga4_pageviews**    | Known because pageview history has already been collected.                         |

**Label / Proxy**

Content Opportunity Score (proxy) — a ranking score used to prioritize which pages should be refreshed or improved.

**Context**


*   report_date
*   month
*   client_hash_id
*   content_hash_id

**Excluded**


*   Client names (not included for privacy)
*   URLs (excluded to avoid exposing client information)
*   Private search queries (not available in this dataset)
*   Future performance metrics (excluded to prevent data leakage)



In [10]:
leak_df = con.execute("""
SELECT
    gsc_clicks,
    gsc_impressions,
    gsc_avg_position,
    ga4_sessions,
    ga4_pageviews,

    -- Honest proxy label
    (gsc_clicks * 1.0 / NULLIF(gsc_impressions,0)) AS opportunity_score,

    -- Deliberate leakage
    (gsc_clicks * 1.0 / NULLIF(gsc_impressions,0)) AS leaked_feature

FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
LIMIT 10
""").df()

leak_df.head()

,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_pageviews,opportunity_score,leaked_feature
0,0,20,3.350000,<NA>,<NA>,0.000,0.000
1,0,1,0.000000,<NA>,<NA>,0.000,0.000
2,1,125,4.928000,<NA>,<NA>,0.008,0.008
3,0,7,4.000000,<NA>,<NA>,0.000,0.000
4,0,11,2.272727,<NA>,<NA>,0.000,0.000


**Leakage result**

The leaked feature was derived directly from the proxy label, making it unsuitable for training because it reveals the target information. This would artificially inflate model performance. I removed the leaked feature and retained only features that would be available at the decision time.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verification

I will verify that:

1. One row represents one webpage.
2. March 2026 data exists.
3. Only rows with available search data are used.

In [7]:
con.execute("""
SELECT
COUNT(*) total_rows,
COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) unique_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows
0,9841378,9841378


In [8]:
con.execute("""
SELECT
COUNT(*) AS rows,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""").df()

,rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [9]:
con.execute("""
SELECT COUNT(*) AS available_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


Leakage experiment

I deliberately added a label-derived feature to demonstrate data leakage. Using future information makes model performance appear unrealistically good. After observing this, I removed the feature and kept the honest version of the dataset.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limitations

This dataset does not contain private client information or actual business outcomes.

The labels are proxy measures rather than true business value.

Some historical data may only contain Google Search Console metrics.

Because this is observational data, the results support decision-making rather than proving causation.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.